# Eval: medir resultados

Inputs y expectativas → ejecución y comparación exacta → informe de casos.

API **2.1.3**. Cada ejemplo muestra una entrada y su resultado. Python Runtime realiza trabajo determinista.

In [ ]:
import agentic_systems as toolkit


In [ ]:
@toolkit.tool
def increment(value: int) -> dict:
    """Suma uno de forma determinista."""
    return {"value": value + 1}

@toolkit.tool
def double(value: int) -> dict:
    """Duplica el valor."""
    return {"value": value * 2}

arithmetic = toolkit.skill(name="arithmetic", tools=[increment, double])
increase = toolkit.Action("increase", "increment")
multiply = toolkit.Action("multiply", "double")

In [ ]:
agent = toolkit.Agent("calculator", toolkit.Sequence((increase, multiply)), skills=(arithmetic,))
cases = ({"input": {"value": 3}, "expected": {"ok": True, "data": {"value": 8}}},
         {"input": {"value": -3}, "expected": {"ok": True, "data": {"value": -4}}})
result = toolkit.Eval(name="contract", unit=agent, cases=cases).run()
assert result.ok and result.data["passed"] == 2
wrong = toolkit.Eval("negative", agent, ({"input": {"value": 3}, "expected": {"data": {"value": 999}}},)).run()
assert not wrong.ok and wrong.data["failed"] == 1 and wrong.children[0].ok
print({"passed": result.data["passed"], "wrong_expectation_detected": not wrong.ok})

Eval puede medir un episodio completo. Un check recibe el RunResult y el caso y devuelve {ok, issues}. El reward de este ejemplo se calcula a partir del total final; no hay un segundo evaluador de Environment.

In [ ]:
import agentic_systems as toolkit

def build_episode(backend='native'):
    worker = toolkit.Agent('sum', toolkit.Action('sum', lambda d: {'total': d['previous'] + d['amount']}), backend=backend)
    return toolkit.Environment('ledger', worker, ({'amount': 2}, {'amount': 3}, {'amount': 100}), memory={'total': 0}, observe=lambda row, memory: {'amount': row['amount'], 'previous': memory['total']}, update=lambda memory, row, result: result.data, stop=lambda memory, result: memory['total'] >= 5)
evaluation = toolkit.Eval('episode',build_episode(),({'input':{},'expected':{'ok':True}},), checks=(lambda result,case: {'ok':result.data['summary']['memory']['total']==5,'issues':[]},))
assert evaluation.run().ok
assert (await evaluation.arun()).ok
assert (await evaluation.compile().arun()).ok
print(evaluation.inspect())

Eval comprueba `expected.ok` y `expected.data` exactos y conserva las ejecuciones evaluadas. Rechaza claves de expectativas desconocidas. Es validación determinista, sin juez LLM. Eval recibe Agent, System o Environment. Los checks y el juez explícito se explican en [contratos](../api/02_contracts_judges.ipynb); sin judge no hay inferencia de evaluación.